# RSB Bilinguality Research

Purpose: inspect Official Railway / RSB articles for Hindi/English output quality, bilingual completeness, fact scores, plagiarism scores, and Hindi information coverage.

This notebook is research-only. It does not modify app files or pipeline outputs.

In [ ]:
import json
import os
import re
from pathlib import Path
from collections import Counter

# Locate the research package by walking up, so this notebook works whether the
# kernel's working directory is research/notebooks/, research/ or the repo root.
def _find_research(start):
    for p in (start, *start.parents):
        for cand in (p, p / 'research'):
            if (cand / 'shared').is_dir() and (cand / 'web_search').is_dir():
                return cand
    raise RuntimeError('cannot locate the research package from %s' % start)

RESEARCH_DIR = _find_research(Path.cwd().resolve())
PROJECT_ROOT = RESEARCH_DIR.parent
OUTPUT_DIR = RESEARCH_DIR / 'output'

AGENCY_FILE = OUTPUT_DIR / 'rsb_agency_output_research.json'
CRAWL_FILE = OUTPUT_DIR / 'rsb_crawl_output_research.json'

data_file = AGENCY_FILE if AGENCY_FILE.exists() else CRAWL_FILE
print('Using file:', data_file)

try:
    articles = json.loads(data_file.read_text(encoding='utf-8'))
except Exception as e:
    print('Could not load data:', e)
    articles = []

print('Total articles:', len(articles))


In [ ]:
# Basic bilinguality and Hindi detection metrics
devanagari_re = re.compile(r'[\u0900-\u097F]')

total = len(articles)
generated = 0
bilingual_complete = 0
source_lang_counts = Counter()
source_has_hindi = 0
hindi_output_articles = 0
hindi_translation_articles = 0

fact_scores_hindi = []
fact_scores_all = []
plag_scores_hindi = []
plag_scores_all = []

for art in articles:
    source_lang = art.get('source_lang', 'unknown')
    source_lang_counts[source_lang] += 1

    body = art.get('body', '') or ''
    title = art.get('title', '') or ''
    story = art.get('ai_news_story', '') or ''
    translated = art.get('ai_news_story_translated', '') or ''

    has_ai_story = bool(story and not story.startswith('[') and not story.startswith('Generation failed'))
    if has_ai_story:
        generated += 1

    if story.strip() and translated.strip():
        bilingual_complete += 1

    if devanagari_re.search(title + ' ' + body):
        source_has_hindi += 1

    story_has_hindi = bool(devanagari_re.search(story))
    translated_has_hindi = bool(devanagari_re.search(translated))

    if story_has_hindi:
        hindi_output_articles += 1
    if translated_has_hindi:
        hindi_translation_articles += 1

    fact = art.get('fact_score')
    plag = art.get('plagiarism_score')
    if isinstance(fact, (int, float)):
        fact_scores_all.append(fact)
        if source_lang == 'hi' or story_has_hindi or translated_has_hindi:
            fact_scores_hindi.append(fact)
    if isinstance(plag, (int, float)):
        plag_scores_all.append(plag)
        if source_lang == 'hi' or story_has_hindi or translated_has_hindi:
            plag_scores_hindi.append(plag)

def avg(values):
    return round(sum(values) / len(values), 2) if values else None

summary = {
    'total_articles': total,
    'generated_articles': generated,
    'bilingual_complete_articles': bilingual_complete,
    'bilingual_complete_pct': round((bilingual_complete / total) * 100, 2) if total else 0,
    'source_lang_counts': dict(source_lang_counts),
    'source_has_hindi_articles': source_has_hindi,
    'hindi_in_ai_story_articles': hindi_output_articles,
    'hindi_in_translation_articles': hindi_translation_articles,
    'avg_fact_score_all': avg(fact_scores_all),
    'avg_fact_score_hindi_related': avg(fact_scores_hindi),
    'avg_plagiarism_score_all': avg(plag_scores_all),
    'avg_plagiarism_score_hindi_related': avg(plag_scores_hindi),
}

for k, v in summary.items():
    print(f'{k}: {v}')

In [ ]:
# Hindi information compilation: show Hindi-related articles and their scores
rows = []

for i, art in enumerate(articles):
    title = art.get('title', '') or ''
    body = art.get('body', '') or ''
    story = art.get('ai_news_story', '') or ''
    translated = art.get('ai_news_story_translated', '') or ''
    text_bundle = ' '.join([title, body, story, translated])

    if devanagari_re.search(text_bundle) or art.get('source_lang') == 'hi':
        rows.append({
            'idx': i,
            'zone': art.get('zone_code', ''),
            'date': art.get('date', ''),
            'source_lang': art.get('source_lang', ''),
            'fact_score': art.get('fact_score', ''),
            'plagiarism_score': art.get('plagiarism_score', ''),
            'story_has_hindi': bool(devanagari_re.search(story)),
            'translation_has_hindi': bool(devanagari_re.search(translated)),
            'title': title[:120],
        })

print('Hindi-related articles:', len(rows))
for row in rows[:25]:
    print(row)